# 06 - DGD Ablation Study

Isolate the two proposed DGD components while keeping the MovieLens-1M protocol,
feature pipeline, training budget, threshold policy, and Cold/Warm A/B/C
evaluation fixed.

## Notebook Linkage and Work Plan

**Input from notebook 02:** the `ml1m-coldstart-v1` protocol bundle. Notebook 06
does not consume model outputs from notebooks 03-05; those are comparison
inputs for the later results notebooks. This notebook reuses the notebook-05
DGD feature/model pattern and changes only graph construction.

**Ablations:**
1. `emerg_exact_topk`: exact graph powers with hard Top-K selection.
2. `mix_topk`: learnable graph-power mixing with hard Top-K selection.
3. `exact_sparsemax`: exact graph powers with row-wise sparsemax.
4. `full_dgd`: learnable graph-power mixing with row-wise sparsemax.

**Outputs:** one immutable versioned ablation bundle per target seed containing
metrics, thresholds, evaluation predictions, structural diagnostics, diffusion
weights, that seed's validation-based advancement decision, and final checkpoints.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import random
import re
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "torch", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 06 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for model notebooks."
    )

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch import nn


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def reject_json_constant(value: str) -> None:
    raise ValueError(f"Non-finite JSON constant is not allowed: {value}")


def strict_json_loads(payload: str | bytes) -> Any:
    return json.loads(payload, parse_constant=reject_json_constant)


def strict_json_dumps(value: Any, **kwargs: Any) -> str:
    return json.dumps(value, allow_nan=False, **kwargs)


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "dgd-ablation-v1"
FAST_DEV_RUN = os.environ.get("COLDSTART_FAST_DEV_RUN", "0") == "1"

requested_device = os.environ.get("COLDSTART_DEVICE")
if requested_device:
    DEVICE = torch.device(requested_device)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("COLDSTART_DEVICE requests CUDA, but torch.cuda is unavailable")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_SEEDS = [2025, 7788, 9999, 3407, 4517]

base_epochs = int(os.environ.get("COLDSTART_ABLATION_EPOCHS", "3"))
base_sample_rows = int(os.environ.get("COLDSTART_ABLATION_SAMPLE_ROWS", "196608"))
base_warm_steps = int(os.environ.get("COLDSTART_ABLATION_WARM_STEPS", "2"))
if FAST_DEV_RUN:
    base_epochs = min(base_epochs, 1)
    base_sample_rows = min(base_sample_rows, 24576)
    base_warm_steps = min(base_warm_steps, 1)

SHARED_MODEL_CONFIG: dict[str, Any] = {
    "embedding_dim": 16,
    "hidden_dim": 64,
    "gnn_layers": 2,
    "learning_rate": 0.001,
    "warm_learning_rate": 0.01,
    "weight_decay": 1e-6,
    "sparsemax_scale": 5.0,
    "topk_edges": 4,
    "epochs": base_epochs,
    "epoch_sample_rows": base_sample_rows,
    "batch_size": 4096,
    "warm_steps": base_warm_steps,
}
ABLATION_MATRIX = pd.DataFrame(
    [
        {
            "variant": "emerg_exact_topk",
            "mix_orders": False,
            "sparsifier": "topk",
            "description": "Exact graph powers with hard Top-K selection; no learnable diffusion.",
        },
        {
            "variant": "mix_topk",
            "mix_orders": True,
            "sparsifier": "topk",
            "description": "Learnable graph-power mixing with hard Top-K selection.",
        },
        {
            "variant": "exact_sparsemax",
            "mix_orders": False,
            "sparsifier": "sparsemax",
            "description": "Exact graph powers with differentiable row-wise sparsemax.",
        },
        {
            "variant": "full_dgd",
            "mix_orders": True,
            "sparsifier": "sparsemax",
            "description": "Learnable graph-power mixing with differentiable row-wise sparsemax.",
        },
    ]
)
RUN_CONFIG_BASE: dict[str, Any] = {
    "schema_version": "dgd-ablation-v1",
    "fast_dev_run": FAST_DEV_RUN,
    "phase_order": ["Cold", "Warm A", "Warm B", "Warm C"],
    "field_order": [
        "user_id", "gender", "age", "occupation", "zip_code",
        "item_id", "release_year", "genres", "title",
    ],
    "item_graph_fields": ["item_id", "release_year", "genres", "title"],
    "max_title_tokens": 8,
    "max_genre_tokens": 6,
    "shared_model_config": SHARED_MODEL_CONFIG,
    "variants": ABLATION_MATRIX.to_dict(orient="records"),
    "selection_rule": "advance the variant with highest mean validation F1; evaluation metrics are reported only",
}


def make_run_config(seed: int) -> dict[str, Any]:
    return {**RUN_CONFIG_BASE, "seed": int(seed)}


def run_config_hash(run_config: dict[str, Any]) -> str:
    return hashlib.sha256(
        strict_json_dumps(run_config, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()


def protocol_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_PROTOCOL_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / PROTOCOL_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "coldstart-v1" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            parent = pointer.parent
            if (
                parent.name == "coldstart-v1"
                and parent.parent.name == "ml-1m"
                and parent.parent.parent.name == "protocols"
            ):
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = strict_json_loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1":
        raise ValueError(f"Unexpected protocol schema: {manifest.get('protocol_schema_version')!r}")
    if manifest.get("protocol_status") != "PASS":
        raise ValueError(f"Protocol status is not PASS: {manifest.get('protocol_status')!r}")
    checks = manifest.get("checks")
    if not isinstance(checks, list) or not checks or not all(
        isinstance(row, dict) and row.get("status") == "PASS" for row in checks
    ):
        raise ValueError("One or more notebook-02 protocol checks did not pass")

    bundle_id = manifest.get("bundle_id")
    if not isinstance(bundle_id, str) or not bundle_id:
        raise ValueError("Protocol bundle_id must be a nonempty string")
    expected_pointer = (root / PROTOCOL_RELATIVE_MANIFEST).resolve()
    if pointer.resolve() != expected_pointer:
        raise ValueError("Protocol pointer is not at the canonical relative path")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    generation_root = (
        root / PROTOCOL_RELATIVE_MANIFEST.parent / "generations" / bundle_id
    ).resolve()
    if bundle_manifest != generation_root / "manifest.json":
        raise ValueError("Protocol bundle_manifest is not the canonical immutable path")
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Protocol pointer and immutable generation manifest differ")

    artifacts = manifest.get("artifacts")
    output_schemas = manifest.get("output_schemas")
    if not isinstance(artifacts, dict) or not artifacts:
        raise ValueError("Protocol manifest has no artifacts")
    if not isinstance(output_schemas, dict) or set(output_schemas) != set(artifacts):
        raise ValueError("Protocol artifact and output-schema contracts differ")

    tables: dict[str, pd.DataFrame] = {}
    for name, artifact in artifacts.items():
        if not isinstance(artifact, dict) or not isinstance(artifact.get("rows"), int):
            raise ValueError(f"Protocol artifact metadata is invalid: {name}")
        path = resolve_inside(root, artifact["path"])
        path.relative_to(generation_root)
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Protocol artifact verification failed: {name}")
        schema = output_schemas[name]
        if (
            not isinstance(schema, dict)
            or not isinstance(schema.get("columns"), list)
            or not schema["columns"]
            or set(schema.get("read_csv_dtypes", {})) != set(schema["columns"])
        ):
            raise ValueError(f"Protocol output schema is invalid: {name}")
        table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Protocol table contract failed: {name}")
        tables[name] = table
    return manifest, tables


PROTOCOL_ERRORS: list[str] = []
PROTOCOL_ROOT = None
PROTOCOL_POINTER = None
PROTOCOL_MANIFEST = None
TABLES = None
for candidate_root, candidate_pointer in protocol_candidates():
    try:
        PROTOCOL_MANIFEST, TABLES = load_verified_protocol(candidate_root, candidate_pointer)
        PROTOCOL_ROOT, PROTOCOL_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        PROTOCOL_ERRORS.append(f"{candidate_pointer}: {error}")

if PROTOCOL_MANIFEST is None or TABLES is None or PROTOCOL_POINTER is None:
    raise RuntimeError(
        "No valid notebook-02 protocol bundle found. Set COLDSTART_PROTOCOL_ROOT. "
        + " | ".join(PROTOCOL_ERRORS)
    )

TUNING_TRAIN = TABLES["tuning_train"]
FINAL_TRAIN = TABLES["final_train"]
VALIDATION_TASKS = TABLES["validation_tasks"]
EVALUATION_TASKS = TABLES["evaluation_tasks"]
USERS = TABLES["users"].sort_values("user_idx").reset_index(drop=True)
ITEMS = TABLES["items"].sort_values("item_idx").reset_index(drop=True)
N_USERS = int(USERS["user_idx"].max()) + 1
N_ITEMS = int(ITEMS["item_idx"].max()) + 1
PHASES = tuple(RUN_CONFIG_BASE["phase_order"])
PHASE_INCREMENT_ROLE = {"Warm A": "warm_a", "Warm B": "warm_b", "Warm C": "warm_c"}
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(DEVICE),
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "variants": len(ABLATION_MATRIX),
            "validation_query_rows": int(VALIDATION_TASKS["role"].eq("query").sum()),
            "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
            "target_seeds": TARGET_SEEDS,
            "run_config_sha256_by_seed": {
                seed: run_config_hash(make_run_config(seed)) for seed in TARGET_SEEDS
            },
        }
    ]
)
display(ABLATION_MATRIX)

,execution_context,python,torch,device,protocol_bundle,variants,validation_query_rows,evaluation_query_rows,target_seeds,run_config_sha256_by_seed
0,kaggle,3.12.13,2.10.0+cu128,cuda,20260716T201434-04e93786b698,4,152762,57569,"[2025, 7788, 9999, 3407, 4517]",{2025: '7a957fa3715a205c13a264e5f87741c7c670d4...


,variant,mix_orders,sparsifier,description
0,emerg_exact_topk,False,topk,Exact graph powers with hard Top-K selection; ...
1,mix_topk,True,topk,Learnable graph-power mixing with hard Top-K s...
2,exact_sparsemax,False,sparsemax,Exact graph powers with differentiable row-wis...
3,full_dgd,True,sparsemax,Learnable graph-power mixing with differentiab...


In [2]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<PAD>"
UNK = "<UNK>"


def make_vocab(values: Iterable[Any]) -> dict[str, int]:
    vocab = {PAD: 0, UNK: 1}
    for value in sorted({str(item) for item in values if pd.notna(item)}):
        if value and value not in vocab:
            vocab[value] = len(vocab)
    return vocab


def encode_scalar(vocab: dict[str, int], value: Any) -> int:
    return vocab.get(str(value), vocab[UNK]) if pd.notna(value) else vocab[UNK]


def title_tokens(title: Any) -> list[str]:
    return TOKEN_RE.findall(str(title).lower())


def genre_tokens(genres: Any) -> list[str]:
    return [token for token in str(genres).split("|") if token]


def encode_sequence(vocab: dict[str, int], tokens: list[str], max_len: int) -> list[int]:
    encoded = [vocab.get(token, vocab[UNK]) for token in tokens[:max_len]]
    return encoded + [vocab[PAD]] * (max_len - len(encoded))


class FeatureStore:
    def __init__(self, users: pd.DataFrame, items: pd.DataFrame, train: pd.DataFrame):
        train_user_ids = set(train["user_idx"].astype(int))
        train_item_ids = set(train["item_idx"].astype(int))
        train_users = users[users["user_idx"].isin(train_user_ids)]
        train_items = items[items["item_idx"].isin(train_item_ids)]

        self.gender_vocab = make_vocab(train_users["gender"])
        self.age_vocab = make_vocab(train_users["age"])
        self.occupation_vocab = make_vocab(train_users["occupation"])
        self.zip_vocab = make_vocab(train_users["zip_code"])
        self.genre_vocab = make_vocab(
            token for value in train_items["genres"] for token in genre_tokens(value)
        )
        self.title_vocab = make_vocab(
            token for value in train_items["title"] for token in title_tokens(value)
        )

        train_years = pd.to_numeric(train_items["release_year"], errors="coerce").astype("float32")
        self.release_mean = float(train_years.mean())
        self.release_std = float(train_years.std() if train_years.std() > 0 else 1.0)

        users = users.sort_values("user_idx").reset_index(drop=True)
        items = items.sort_values("item_idx").reset_index(drop=True)
        self.user_gender = np.array([encode_scalar(self.gender_vocab, x) for x in users["gender"]], dtype=np.int64)
        self.user_age = np.array([encode_scalar(self.age_vocab, x) for x in users["age"]], dtype=np.int64)
        self.user_occupation = np.array([encode_scalar(self.occupation_vocab, x) for x in users["occupation"]], dtype=np.int64)
        self.user_zip = np.array([encode_scalar(self.zip_vocab, x) for x in users["zip_code"]], dtype=np.int64)

        years = pd.to_numeric(items["release_year"], errors="coerce").astype("float32")
        years = years.fillna(self.release_mean)
        self.item_release = ((years.to_numpy(dtype=np.float32) - self.release_mean) / self.release_std).astype(np.float32)
        self.item_genres = np.array(
            [
                encode_sequence(
                    self.genre_vocab, genre_tokens(value), int(RUN_CONFIG_BASE["max_genre_tokens"])
                )
                for value in items["genres"]
            ],
            dtype=np.int64,
        )
        self.item_titles = np.array(
            [
                encode_sequence(
                    self.title_vocab, title_tokens(value), int(RUN_CONFIG_BASE["max_title_tokens"])
                )
                for value in items["title"]
            ],
            dtype=np.int64,
        )
        self.field_sizes = {
            "user_id": N_USERS,
            "gender": len(self.gender_vocab),
            "age": len(self.age_vocab),
            "occupation": len(self.occupation_vocab),
            "zip_code": len(self.zip_vocab),
            "item_id": N_ITEMS,
            "genres": len(self.genre_vocab),
            "title": len(self.title_vocab),
        }

    def arrays(self, table: pd.DataFrame) -> dict[str, np.ndarray]:
        return {
            "source_row": table["source_row"].to_numpy(dtype=np.int64),
            "user_id": table["user_id"].to_numpy(dtype=np.int64),
            "user_idx": table["user_idx"].to_numpy(dtype=np.int64),
            "item_id": table["item_id"].to_numpy(dtype=np.int64),
            "item_idx": table["item_idx"].to_numpy(dtype=np.int64),
            "label": table["label"].to_numpy(dtype=np.float32),
        }

    def batch(self, arrays: dict[str, np.ndarray], rows: np.ndarray) -> tuple[dict[str, torch.Tensor], torch.Tensor]:
        user_idx = arrays["user_idx"][rows]
        item_idx = arrays["item_idx"][rows]
        features = {
            "user_id": torch.as_tensor(user_idx, dtype=torch.long, device=DEVICE),
            "gender": torch.as_tensor(self.user_gender[user_idx], dtype=torch.long, device=DEVICE),
            "age": torch.as_tensor(self.user_age[user_idx], dtype=torch.long, device=DEVICE),
            "occupation": torch.as_tensor(self.user_occupation[user_idx], dtype=torch.long, device=DEVICE),
            "zip_code": torch.as_tensor(self.user_zip[user_idx], dtype=torch.long, device=DEVICE),
            "item_id": torch.as_tensor(item_idx, dtype=torch.long, device=DEVICE),
            "release_year": torch.as_tensor(self.item_release[item_idx], dtype=torch.float32, device=DEVICE),
            "genres": torch.as_tensor(self.item_genres[item_idx], dtype=torch.long, device=DEVICE),
            "title": torch.as_tensor(self.item_titles[item_idx], dtype=torch.long, device=DEVICE),
        }
        labels = torch.as_tensor(arrays["label"][rows], dtype=torch.float32, device=DEVICE)
        return features, labels

    def contract(self) -> dict[str, Any]:
        vocabs = {
            "gender": self.gender_vocab,
            "age": self.age_vocab,
            "occupation": self.occupation_vocab,
            "zip_code": self.zip_vocab,
            "genres": self.genre_vocab,
            "title": self.title_vocab,
        }
        return {
            "schema_version": "dgd-ablation-feature-contract-v1",
            "field_order": RUN_CONFIG_BASE["field_order"],
            "item_graph_fields": RUN_CONFIG_BASE["item_graph_fields"],
            "max_title_tokens": RUN_CONFIG_BASE["max_title_tokens"],
            "max_genre_tokens": RUN_CONFIG_BASE["max_genre_tokens"],
            "release_year_mean": self.release_mean,
            "release_year_std": self.release_std,
            "field_sizes": self.field_sizes,
            "vocab_sha256": {
                name: sha256_bytes(strict_json_dumps(vocab, sort_keys=True).encode())
                for name, vocab in vocabs.items()
            },
            "vocab_sizes": {name: len(vocab) for name, vocab in vocabs.items()},
            "vocabs": vocabs,
        }


FEATURES = FeatureStore(USERS, ITEMS, TUNING_TRAIN)
FEATURE_CONTRACT = FEATURES.contract()
display(pd.DataFrame([FEATURE_CONTRACT["field_sizes"]]))
show_records(
    [
        {
            "feature_contract": FEATURE_CONTRACT["schema_version"],
            "title_vocab": FEATURE_CONTRACT["vocab_sizes"]["title"],
            "genre_vocab": FEATURE_CONTRACT["vocab_sizes"]["genres"],
            "zip_vocab": FEATURE_CONTRACT["vocab_sizes"]["zip_code"],
        }
    ]
)

,user_id,gender,age,occupation,zip_code,item_id,genres,title
0,6040,4,9,23,3441,2375,20,1764


,feature_contract,title_vocab,genre_vocab,zip_vocab
0,dgd-ablation-feature-contract-v1,1764,20,3441


In [3]:
FIELD_ORDER = list(RUN_CONFIG_BASE["field_order"])
ITEM_GRAPH_FIELDS = list(RUN_CONFIG_BASE["item_graph_fields"])
ITEM_GRAPH_POSITIONS = [FIELD_ORDER.index(name) for name in ITEM_GRAPH_FIELDS]


def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    shifted = logits - logits.max(dim=dim, keepdim=True).values
    sorted_logits, _ = torch.sort(shifted, descending=True, dim=dim)
    range_values = torch.arange(1, logits.size(dim) + 1, device=logits.device, dtype=logits.dtype)
    view_shape = [1] * logits.dim()
    view_shape[dim] = -1
    range_values = range_values.view(view_shape)
    support = 1 + range_values * sorted_logits > torch.cumsum(sorted_logits, dim=dim)
    support_size = support.sum(dim=dim, keepdim=True).clamp_min(1)
    tau = (
        torch.gather(torch.cumsum(sorted_logits, dim=dim), dim, support_size - 1) - 1
    ) / support_size.to(logits.dtype)
    return torch.clamp(shifted - tau, min=0.0)


def hard_topk_graph(scores: torch.Tensor, topk_edges: int) -> torch.Tensor:
    dense = torch.softmax(scores, dim=-1)
    k = min(topk_edges, dense.shape[-1])
    _, index = torch.topk(dense, k=k, dim=-1)
    mask = torch.zeros_like(dense).scatter_(-1, index, 1.0)
    pruned = dense * mask
    return pruned / pruned.sum(dim=-1, keepdim=True).clamp_min(1e-12)


class AblationCTR(nn.Module):
    def __init__(self, feature_sizes: dict[str, int], config: dict[str, Any], variant: dict[str, Any]):
        super().__init__()
        embedding_dim = int(config["embedding_dim"])
        hidden_dim = int(config["hidden_dim"])
        self.num_fields = len(FIELD_ORDER)
        self.gnn_layers = int(config["gnn_layers"])
        self.sparsemax_scale = float(config["sparsemax_scale"])
        self.topk_edges = int(config["topk_edges"])
        self.mix_orders = bool(variant["mix_orders"])
        self.sparsifier = str(variant["sparsifier"])
        self.variant_name = str(variant["variant"])
        self.embeddings = nn.ModuleDict(
            {
                "user_id": nn.Embedding(feature_sizes["user_id"], embedding_dim),
                "gender": nn.Embedding(feature_sizes["gender"], embedding_dim, padding_idx=0),
                "age": nn.Embedding(feature_sizes["age"], embedding_dim, padding_idx=0),
                "occupation": nn.Embedding(feature_sizes["occupation"], embedding_dim, padding_idx=0),
                "zip_code": nn.Embedding(feature_sizes["zip_code"], embedding_dim, padding_idx=0),
                "item_id": nn.Embedding(feature_sizes["item_id"], embedding_dim),
                "genres": nn.Embedding(feature_sizes["genres"], embedding_dim, padding_idx=0),
                "title": nn.Embedding(feature_sizes["title"], embedding_dim, padding_idx=0),
            }
        )
        self.release_weight = nn.Parameter(torch.empty(embedding_dim))
        self.item_graph_delta = nn.Embedding(feature_sizes["item_id"], self.num_fields * self.num_fields)
        self.graph_generator = nn.Sequential(
            nn.Linear(len(ITEM_GRAPH_FIELDS) * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.num_fields * self.num_fields),
        )
        self.diffusion_logits = nn.Parameter(torch.zeros(self.gnn_layers, self.gnn_layers))
        self.graph_layers = nn.ModuleList(
            [nn.Linear(embedding_dim, embedding_dim, bias=False) for _ in range(self.gnn_layers)]
        )
        self.head = nn.Sequential(
            nn.Linear(self.num_fields * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)
                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
        nn.init.normal_(self.release_weight, std=0.02)
        nn.init.zeros_(self.item_graph_delta.weight)
        nn.init.zeros_(self.diffusion_logits)

    def sequence_embedding(self, name: str, tokens: torch.Tensor) -> torch.Tensor:
        embedded = self.embeddings[name](tokens)
        mask = tokens.ne(0).float().unsqueeze(-1)
        denom = mask.sum(dim=1).clamp_min(1.0)
        return (embedded * mask).sum(dim=1) / denom

    def field_embeddings(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
    ) -> torch.Tensor:
        item_embedding = self.embeddings["item_id"](features["item_id"])
        if local_item_embedding is not None:
            item_embedding = local_item_embedding.unsqueeze(0).expand_as(item_embedding)
        fields = [
            self.embeddings["user_id"](features["user_id"]),
            self.embeddings["gender"](features["gender"]),
            self.embeddings["age"](features["age"]),
            self.embeddings["occupation"](features["occupation"]),
            self.embeddings["zip_code"](features["zip_code"]),
            item_embedding,
            features["release_year"].unsqueeze(1) * self.release_weight.unsqueeze(0),
            self.sequence_embedding("genres", features["genres"]),
            self.sequence_embedding("title", features["title"]),
        ]
        return torch.stack(fields, dim=1)

    def first_order_graph(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        field_emb = self.field_embeddings(features, local_item_embedding)
        item_context = field_emb[:, ITEM_GRAPH_POSITIONS, :].reshape(field_emb.shape[0], -1)
        raw_graph = self.graph_generator(item_context)
        if local_graph_delta is None:
            raw_graph = raw_graph + self.item_graph_delta(features["item_id"])
        else:
            raw_graph = raw_graph + local_graph_delta.unsqueeze(0).expand_as(raw_graph)
        raw_graph = raw_graph.reshape(-1, self.num_fields, self.num_fields)
        raw_graph = 0.5 * (raw_graph + raw_graph.transpose(1, 2))
        return raw_graph / (self.num_fields ** 0.5)

    def normalize_graph(self, scores: torch.Tensor) -> torch.Tensor:
        if self.sparsifier == "sparsemax":
            return sparsemax(scores * self.sparsemax_scale, dim=-1)
        if self.sparsifier == "topk":
            return hard_topk_graph(scores, self.topk_edges)
        raise ValueError(f"Unknown sparsifier: {self.sparsifier}")

    def diffused_graphs(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> tuple[list[torch.Tensor], torch.Tensor]:
        base_logits = self.first_order_graph(features, local_item_embedding, local_graph_delta)
        powers = [base_logits]
        for _ in range(1, self.gnn_layers):
            powers.append(torch.bmm(powers[-1], base_logits) / (self.num_fields ** 0.5))

        graphs: list[torch.Tensor] = []
        for layer_index in range(self.gnn_layers):
            if self.mix_orders:
                weights = torch.softmax(self.diffusion_logits[layer_index, : layer_index + 1], dim=0)
                mixed = sum(weights[power_index] * powers[power_index] for power_index in range(layer_index + 1))
            else:
                mixed = powers[layer_index]
            graphs.append(self.normalize_graph(mixed))
        return graphs, base_logits

    def forward(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        h0 = self.field_embeddings(features, local_item_embedding)
        h = h0
        graphs, _ = self.diffused_graphs(features, local_item_embedding, local_graph_delta)
        for layer, graph in zip(self.graph_layers, graphs):
            message = torch.bmm(graph, layer(h0))
            h = h * (1.0 + torch.tanh(message))
        return self.head(h.reshape(h.shape[0], -1)).squeeze(1)

    def diffusion_weight_rows(self) -> list[dict[str, Any]]:
        rows: list[dict[str, Any]] = []
        for layer_index in range(self.gnn_layers):
            if self.mix_orders:
                weights = torch.softmax(self.diffusion_logits[layer_index, : layer_index + 1], dim=0)
                values = weights.detach().cpu().tolist()
            else:
                values = [0.0] * (layer_index + 1)
                values[layer_index] = 1.0
            for power_index, weight in enumerate(values, start=1):
                rows.append({"variant": self.variant_name, "layer": layer_index + 1, "power": power_index, "weight": float(weight)})
        return rows


def roc_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.arange(1, len(scores) + 1, dtype=np.float64)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        if end - start > 1:
            ranks[start:end] = ranks[start:end].mean()
        start = end
    original_ranks = np.empty_like(ranks)
    original_ranks[order] = ranks
    return float((original_ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    labels = labels.astype(np.int64)
    if labels.size == 0 or scores.size != labels.size:
        raise ValueError("F1 threshold selection requires nonempty aligned labels and scores")
    if not np.isfinite(scores).all():
        raise ValueError("F1 threshold selection received non-finite scores")
    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    sorted_scores = scores[order]
    tp = np.cumsum(sorted_labels)
    fp = np.cumsum(1 - sorted_labels)
    fn = int(sorted_labels.sum()) - tp
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator > 0)
    realizable = np.r_[sorted_scores[:-1] != sorted_scores[1:], True]
    realizable_indices = np.flatnonzero(realizable)
    best = int(realizable_indices[np.argmax(f1[realizable_indices])])
    return float(sorted_scores[best]), float(f1[best])


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    labels = labels.astype(np.int64)
    if labels.size == 0 or scores.size != labels.size:
        raise ValueError("Binary metrics require nonempty aligned labels and scores")
    if not np.isfinite(scores).all() or not np.isfinite(threshold):
        raise ValueError("Binary metrics received a non-finite score or threshold")
    predictions = scores >= threshold
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "rows": int(len(labels)),
        "positives": int(labels.sum()),
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / len(labels)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": roc_auc(labels, scores),
        "predicted_positive_rate": float(predictions.mean()),
        "score_mean": float(scores.mean()),
        "score_std": float(scores.std()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def numeric_values_finite(table: pd.DataFrame) -> bool:
    numeric = table.select_dtypes(include=[np.number])
    return bool(
        len(table) > 0
        and numeric.shape[1] > 0
        and np.isfinite(numeric.to_numpy(dtype=np.float64)).all()
    )


def summarize_predictions(predictions: pd.DataFrame, split: str, thresholds: dict[tuple[str, str], float] | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    threshold_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for variant in ABLATION_MATRIX["variant"]:
        for phase in PHASES:
            phase_predictions = predictions[predictions["variant"].eq(variant) & predictions["phase"].eq(phase)]
            labels = phase_predictions["label"].to_numpy(dtype=np.int64)
            scores = phase_predictions["score"].to_numpy(dtype=np.float64)
            if thresholds is None:
                threshold, best_f1 = best_f1_threshold(labels, scores)
            else:
                threshold, best_f1 = float(thresholds[(variant, phase)]), np.nan
            threshold_rows.append(
                {"split": split, "variant": variant, "phase": phase, "threshold": threshold, "validation_best_f1": best_f1}
            )
            metric_rows.append({"split": split, "variant": variant, "phase": phase, **binary_metrics(labels, scores, threshold)})
    return pd.DataFrame(threshold_rows), pd.DataFrame(metric_rows)

In [4]:
def train_variant_model(train_table: pd.DataFrame, variant: dict[str, Any], seed: int, run_name: str) -> tuple[AblationCTR, pd.DataFrame, float]:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    arrays = FEATURES.arrays(train_table)
    model = AblationCTR(FEATURES.field_sizes, SHARED_MODEL_CONFIG, variant).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=float(SHARED_MODEL_CONFIG["learning_rate"]), weight_decay=float(SHARED_MODEL_CONFIG["weight_decay"])
    )
    history: list[dict[str, Any]] = []
    n_rows = len(arrays["label"])
    sample_rows = min(int(SHARED_MODEL_CONFIG["epoch_sample_rows"]), n_rows)
    batch_size = int(SHARED_MODEL_CONFIG["batch_size"])

    for epoch in range(1, int(SHARED_MODEL_CONFIG["epochs"]) + 1):
        model.train()
        order = rng.choice(n_rows, size=sample_rows, replace=False) if sample_rows < n_rows else rng.permutation(n_rows)
        total_loss = 0.0
        total_examples = 0
        for start_idx in range(0, len(order), batch_size):
            rows = order[start_idx : start_idx + batch_size]
            features, labels = FEATURES.batch(arrays, rows)
            optimizer.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(model(features), labels)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(rows)
            total_examples += len(rows)
        history.append(
            {
                "variant": variant["variant"],
                "run_name": run_name,
                "epoch": epoch,
                "loss": total_loss / max(total_examples, 1),
                "examples": total_examples,
            }
        )
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, pd.DataFrame(history), time.perf_counter() - start


def freeze_parameters(model: nn.Module, value: bool) -> list[bool]:
    previous = [parameter.requires_grad for parameter in model.parameters()]
    for parameter in model.parameters():
        parameter.requires_grad_(value)
    return previous


def restore_requires_grad(model: nn.Module, previous: list[bool]) -> None:
    for parameter, requires_grad in zip(model.parameters(), previous):
        parameter.requires_grad_(requires_grad)


def predict_table(model: AblationCTR, table: pd.DataFrame, local_item_embedding: torch.Tensor | None = None, local_graph_delta: torch.Tensor | None = None, batch_size: int = 8192) -> np.ndarray:
    arrays = FEATURES.arrays(table)
    scores: list[np.ndarray] = []
    model.eval()
    with torch.no_grad():
        for start_idx in range(0, len(table), batch_size):
            rows = np.arange(start_idx, min(start_idx + batch_size, len(table)), dtype=np.int64)
            features, _ = FEATURES.batch(arrays, rows)
            scores.append(torch.sigmoid(model(features, local_item_embedding, local_graph_delta)).detach().cpu().numpy())
    return np.concatenate(scores).astype(np.float32)


def adapt_local_state(model: AblationCTR, support_table: pd.DataFrame, local_item_embedding: torch.Tensor, local_graph_delta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, float]:
    if support_table.empty:
        return local_item_embedding, local_graph_delta, 0.0
    arrays = FEATURES.arrays(support_table.reset_index(drop=True))
    rows = np.arange(len(support_table), dtype=np.int64)
    optimizer = torch.optim.Adam([local_item_embedding, local_graph_delta], lr=float(SHARED_MODEL_CONFIG["warm_learning_rate"]))
    last_loss = 0.0
    model.eval()
    for _ in range(int(SHARED_MODEL_CONFIG["warm_steps"])):
        features, labels = FEATURES.batch(arrays, rows)
        optimizer.zero_grad(set_to_none=True)
        loss = F.binary_cross_entropy_with_logits(model(features, local_item_embedding, local_graph_delta), labels)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
    return local_item_embedding.detach().requires_grad_(), local_graph_delta.detach().requires_grad_(), last_loss


def graph_stats(model: AblationCTR, one_row: pd.DataFrame, phase: str, local_item_embedding: torch.Tensor | None, local_graph_delta: torch.Tensor | None, warm_loss: float) -> dict[str, Any]:
    arrays = FEATURES.arrays(one_row.reset_index(drop=True))
    features, _ = FEATURES.batch(arrays, np.array([0], dtype=np.int64))
    with torch.no_grad():
        graphs, base = model.diffused_graphs(features, local_item_embedding, local_graph_delta)
        final_graph = graphs[-1][0].detach().cpu().numpy()
        base_graph = base[0].detach().cpu().numpy()
    return {
        "variant": model.variant_name,
        "item_id": int(one_row["item_id"].iloc[0]),
        "item_idx": int(one_row["item_idx"].iloc[0]),
        "phase": phase,
        "base_graph_mean": float(base_graph.mean()),
        "graph_density_gt_1e_6": float((final_graph > 1e-6).mean()),
        "graph_row_sum_error": float(np.abs(final_graph.sum(axis=1) - 1.0).max()),
        "local_delta_norm": float(local_graph_delta.detach().norm().cpu()) if local_graph_delta is not None else 0.0,
        "last_warm_loss": float(warm_loss),
    }


def score_tasks_with_warmup(model: AblationCTR, tasks: pd.DataFrame, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions: list[pd.DataFrame] = []
    diagnostics: list[dict[str, Any]] = []
    previous_requires_grad = freeze_parameters(model, False)
    try:
        sorted_tasks = tasks.sort_values(["item_idx", "item_rank", "source_row"]).reset_index(drop=True)
        for item_idx, item_rows in sorted_tasks.groupby("item_idx", sort=True):
            item_rows = item_rows.reset_index(drop=True)
            query_rows = item_rows[item_rows["role"].eq("query")].reset_index(drop=True)
            local_embedding = model.embeddings["item_id"].weight[int(item_idx)].detach().clone().requires_grad_()
            local_delta = model.item_graph_delta.weight[int(item_idx)].detach().clone().requires_grad_()

            cold_predictions = query_rows[["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]].copy()
            cold_predictions.insert(0, "phase", "Cold")
            cold_predictions.insert(0, "variant", model.variant_name)
            cold_predictions["score"] = predict_table(model, query_rows)
            predictions.append(cold_predictions)
            diagnostics.append(graph_stats(model, query_rows.iloc[[0]], "Cold", None, None, 0.0))

            for phase, role in PHASE_INCREMENT_ROLE.items():
                support_rows = item_rows[item_rows["role"].eq(role)].reset_index(drop=True)
                local_embedding, local_delta, warm_loss = adapt_local_state(model, support_rows, local_embedding, local_delta)
                phase_predictions = query_rows[["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]].copy()
                phase_predictions.insert(0, "phase", phase)
                phase_predictions.insert(0, "variant", model.variant_name)
                phase_predictions["score"] = predict_table(model, query_rows, local_embedding, local_delta)
                predictions.append(phase_predictions)
                diagnostics.append(graph_stats(model, query_rows.iloc[[0]], phase, local_embedding, local_delta, warm_loss))
    finally:
        restore_requires_grad(model, previous_requires_grad)
    prediction_table = pd.concat(predictions, ignore_index=True)
    diagnostics_table = pd.DataFrame(diagnostics)
    diagnostics_table.insert(0, "split", split)
    return prediction_table, diagnostics_table


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def variant_gradient_check(variant: dict[str, Any], seed: int) -> dict[str, Any]:
    seed_everything(seed)
    model = AblationCTR(FEATURES.field_sizes, SHARED_MODEL_CONFIG, variant).to(DEVICE)
    arrays = FEATURES.arrays(TUNING_TRAIN.iloc[: min(256, len(TUNING_TRAIN))])
    rows = np.arange(len(arrays["label"]), dtype=np.int64)
    features, labels = FEATURES.batch(arrays, rows)
    graphs, base = model.diffused_graphs(features)
    loss = F.binary_cross_entropy_with_logits(model(features), labels) + sum(graph.mean() for graph in graphs) * 1e-3
    model.zero_grad(set_to_none=True)
    loss.backward()
    graph_grad_norm = sum(
        float(parameter.grad.detach().abs().sum().cpu())
        for parameter in model.graph_generator.parameters()
        if parameter.grad is not None
    )
    diffusion_grad_norm = float(model.diffusion_logits.grad.detach().abs().sum().cpu()) if model.diffusion_logits.grad is not None else 0.0
    row_sums = torch.cat([graph.sum(dim=-1).detach().flatten() for graph in graphs])
    sparse_fraction = torch.cat([(graph <= 1e-6).float().detach().flatten() for graph in graphs]).mean()
    requires_diffusion_grad = bool(variant["mix_orders"])
    status = (
        graph_grad_norm > 0
        and (diffusion_grad_norm > 0 or not requires_diffusion_grad)
        and float((row_sums - 1.0).abs().max().cpu()) < 1e-4
        and torch.isfinite(base).all().item()
    )
    return {
        "variant": variant["variant"],
        "graph_generator_grad_norm": graph_grad_norm,
        "diffusion_grad_norm": diffusion_grad_norm,
        "requires_diffusion_grad": requires_diffusion_grad,
        "row_sum_error": float((row_sums - 1.0).abs().max().cpu()),
        "sparse_fraction": float(sparse_fraction.cpu()),
        "status": "PASS" if status else "FAIL",
    }

In [5]:
def build_seed_run(target_seed: int) -> dict[str, Any]:
    run_config = make_run_config(target_seed)
    tuning_seed = target_seed + 1_000
    final_seed = target_seed + 10_000
    variants = ABLATION_MATRIX.to_dict(orient="records")
    seed_everything(target_seed)

    gradient_checks = pd.DataFrame(
        [variant_gradient_check(variant, target_seed) for variant in variants]
    )
    training_history_parts: list[pd.DataFrame] = []
    validation_prediction_parts: list[pd.DataFrame] = []
    validation_diagnostic_parts: list[pd.DataFrame] = []
    tuning_seeds_used: list[int] = []

    for variant in variants:
        variant_name = str(variant["variant"])
        tuning_seeds_used.append(tuning_seed)
        tuning_model, tuning_history, tuning_seconds = train_variant_model(
            TUNING_TRAIN,
            variant,
            seed=tuning_seed,
            run_name=f"tuning_{variant_name}",
        )
        training_history_parts.append(tuning_history)
        validation_predictions, validation_diagnostics = score_tasks_with_warmup(
            tuning_model, VALIDATION_TASKS, split="validation"
        )
        validation_prediction_parts.append(validation_predictions)
        validation_diagnostics["tuning_seconds"] = float(tuning_seconds)
        validation_diagnostic_parts.append(validation_diagnostics)
        del tuning_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    validation_predictions = pd.concat(validation_prediction_parts, ignore_index=True)
    validation_thresholds, validation_metrics = summarize_predictions(
        validation_predictions, split="validation"
    )
    threshold_by_variant_phase = {
        (row.variant, row.phase): float(row.threshold)
        for row in validation_thresholds.itertuples(index=False)
    }
    validation_diagnostics = pd.concat(validation_diagnostic_parts, ignore_index=True)
    validation_summary = (
        validation_metrics.groupby("variant", observed=True)
        .agg(
            mean_validation_f1=("f1", "mean"),
            mean_validation_auc=("roc_auc", "mean"),
        )
        .reset_index()
        .sort_values(
            ["mean_validation_f1", "mean_validation_auc"], ascending=False
        )
        .reset_index(drop=True)
    )

    # Freeze the validation-only decision before final refits or evaluation scoring.
    advance_variant = str(validation_summary.loc[0, "variant"])
    selection_decision = pd.DataFrame(
        [
            {
                "advance_to_multiseed": advance_variant,
                "selection_metric": "mean_validation_f1",
                "selection_policy": run_config["selection_rule"],
                "uses_evaluation_for_selection": False,
            }
        ]
    )
    frozen_selection = selection_decision.to_dict(orient="records")

    evaluation_prediction_parts: list[pd.DataFrame] = []
    evaluation_diagnostic_parts: list[pd.DataFrame] = []
    diffusion_weight_parts: list[pd.DataFrame] = []
    final_state_dicts: dict[str, Any] = {}
    config_summary_rows: list[dict[str, Any]] = []
    final_seeds_used: list[int] = []

    for variant in variants:
        variant_name = str(variant["variant"])
        final_seeds_used.append(final_seed)
        final_model, final_history, final_seconds = train_variant_model(
            FINAL_TRAIN,
            variant,
            seed=final_seed,
            run_name=f"final_refit_{variant_name}",
        )
        training_history_parts.append(final_history)
        evaluation_predictions, evaluation_diagnostics = score_tasks_with_warmup(
            final_model, EVALUATION_TASKS, split="evaluation"
        )
        evaluation_diagnostics["final_training_seconds"] = float(final_seconds)
        evaluation_prediction_parts.append(evaluation_predictions)
        evaluation_diagnostic_parts.append(evaluation_diagnostics)
        diffusion_weight_parts.append(pd.DataFrame(final_model.diffusion_weight_rows()))
        final_state_dicts[variant_name] = {
            key: value.detach().cpu() for key, value in final_model.state_dict().items()
        }
        config_summary_rows.append(
            {"variant": variant_name, **SHARED_MODEL_CONFIG, **variant}
        )
        del final_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    training_history = pd.concat(training_history_parts, ignore_index=True)
    evaluation_predictions = pd.concat(evaluation_prediction_parts, ignore_index=True)
    _, evaluation_metrics = summarize_predictions(
        evaluation_predictions,
        split="evaluation",
        thresholds=threshold_by_variant_phase,
    )
    structural_diagnostics = pd.concat(
        [validation_diagnostics, *evaluation_diagnostic_parts], ignore_index=True
    )
    structural_diagnostics["device"] = str(DEVICE)
    diffusion_weights = pd.concat(diffusion_weight_parts, ignore_index=True)
    config_summary = pd.DataFrame(config_summary_rows)
    ablation_summary = (
        validation_summary.set_index("variant")
        .join(
            evaluation_metrics.groupby("variant", observed=True).agg(
                mean_evaluation_f1=("f1", "mean"),
                mean_evaluation_auc=("roc_auc", "mean"),
            )
        )
        .reset_index()
        .sort_values(
            ["mean_validation_f1", "mean_validation_auc"], ascending=False
        )
        .reset_index(drop=True)
    )

    pre_export_checks: list[dict[str, Any]] = []

    def pre_export_check(
        name: str, condition: bool, observed: Any, expected: Any
    ) -> None:
        pre_export_checks.append(
            {
                "check": name,
                "status": "PASS" if condition else "FAIL",
                "observed": observed,
                "expected": expected,
            }
        )

    expected_variants = set(ABLATION_MATRIX["variant"])
    expected_validation_predictions = int(
        VALIDATION_TASKS["role"].eq("query").sum()
        * len(PHASES)
        * len(ABLATION_MATRIX)
    )
    expected_evaluation_predictions = int(
        EVALUATION_TASKS["role"].eq("query").sum()
        * len(PHASES)
        * len(ABLATION_MATRIX)
    )
    final_excludes_evaluation_items = set(FINAL_TRAIN["item_idx"]).isdisjoint(
        set(EVALUATION_TASKS["item_idx"])
    )
    feature_contract_has_pad_unk = all(
        vocab.get(PAD) == 0 and vocab.get(UNK) == 1
        for vocab in [
            FEATURES.gender_vocab,
            FEATURES.age_vocab,
            FEATURES.occupation_vocab,
            FEATURES.zip_vocab,
            FEATURES.genre_vocab,
            FEATURES.title_vocab,
        ]
    )
    selection_matches_validation = advance_variant == str(
        validation_summary.loc[0, "variant"]
    )
    selection_remained_frozen = (
        selection_decision.to_dict(orient="records") == frozen_selection
    )

    pre_export_check(
        "protocol schema",
        PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1",
        PROTOCOL_MANIFEST["protocol_schema_version"],
        "ml1m-coldstart-v1",
    )
    pre_export_check(
        "manifest target seed",
        run_config["seed"] == target_seed,
        run_config["seed"],
        target_seed,
    )
    pre_export_check(
        "all gradient checks pass",
        gradient_checks["status"].eq("PASS").all(),
        gradient_checks["status"].tolist(),
        ["PASS"] * len(ABLATION_MATRIX),
    )
    pre_export_check(
        "variants complete",
        set(ablation_summary["variant"]) == expected_variants,
        sorted(ablation_summary["variant"]),
        sorted(expected_variants),
    )
    pre_export_check(
        "controlled tuning seed across variants",
        tuning_seeds_used == [tuning_seed] * len(variants),
        tuning_seeds_used,
        [tuning_seed] * len(variants),
    )
    pre_export_check(
        "controlled final seed across variants",
        final_seeds_used == [final_seed] * len(variants),
        final_seeds_used,
        [final_seed] * len(variants),
    )
    pre_export_check(
        "validation predictions complete",
        len(validation_predictions) == expected_validation_predictions,
        len(validation_predictions),
        expected_validation_predictions,
    )
    pre_export_check(
        "evaluation predictions complete",
        len(evaluation_predictions) == expected_evaluation_predictions,
        len(evaluation_predictions),
        expected_evaluation_predictions,
    )
    pre_export_check(
        "thresholds complete",
        len(validation_thresholds) == len(ABLATION_MATRIX) * len(PHASES),
        len(validation_thresholds),
        len(ABLATION_MATRIX) * len(PHASES),
    )
    pre_export_check(
        "training losses finite",
        np.isfinite(training_history["loss"].to_numpy(dtype=np.float64)).all(),
        bool(np.isfinite(training_history["loss"].to_numpy(dtype=np.float64)).all()),
        True,
    )
    pre_export_check(
        "validation prediction scores finite",
        np.isfinite(validation_predictions["score"].to_numpy(dtype=np.float64)).all(),
        bool(np.isfinite(validation_predictions["score"].to_numpy(dtype=np.float64)).all()),
        True,
    )
    pre_export_check(
        "evaluation prediction scores finite",
        np.isfinite(evaluation_predictions["score"].to_numpy(dtype=np.float64)).all(),
        bool(np.isfinite(evaluation_predictions["score"].to_numpy(dtype=np.float64)).all()),
        True,
    )
    pre_export_check(
        "validation thresholds finite",
        numeric_values_finite(validation_thresholds),
        numeric_values_finite(validation_thresholds),
        True,
    )
    pre_export_check(
        "validation and evaluation metrics finite",
        numeric_values_finite(validation_metrics) and numeric_values_finite(evaluation_metrics),
        {
            "validation": numeric_values_finite(validation_metrics),
            "evaluation": numeric_values_finite(evaluation_metrics),
        },
        {"validation": True, "evaluation": True},
    )
    pre_export_check(
        "final train excludes evaluation items",
        final_excludes_evaluation_items,
        final_excludes_evaluation_items,
        True,
    )
    pre_export_check(
        "feature contract has PAD/UNK",
        feature_contract_has_pad_unk,
        feature_contract_has_pad_unk,
        True,
    )
    pre_export_check(
        "selection matches validation summary",
        selection_matches_validation,
        advance_variant,
        str(validation_summary.loc[0, "variant"]),
    )
    pre_export_check(
        "selection frozen before evaluation",
        selection_remained_frozen,
        selection_decision.to_dict(orient="records"),
        frozen_selection,
    )
    pre_export_check(
        "selection avoids evaluation metrics",
        not bool(selection_decision["uses_evaluation_for_selection"].iloc[0]),
        False,
        False,
    )
    pre_export_check(
        "state dicts complete",
        set(final_state_dicts) == expected_variants,
        sorted(final_state_dicts),
        sorted(expected_variants),
    )

    if not all(row["status"] == "PASS" for row in pre_export_checks):
        display(pd.DataFrame(pre_export_checks))
        raise RuntimeError(
            f"Ablation checks failed for seed {target_seed}; artifacts were not published"
        )

    output_tables = {
        "ablation_matrix": ABLATION_MATRIX,
        "training_history": training_history,
        "config_summary": config_summary,
        "gradient_checks": gradient_checks,
        "validation_thresholds": validation_thresholds,
        "validation_metrics": validation_metrics,
        "validation_summary": validation_summary,
        "evaluation_metrics": evaluation_metrics,
        "evaluation_predictions": evaluation_predictions,
        "structural_diagnostics": structural_diagnostics,
        "diffusion_weights": diffusion_weights,
        "ablation_summary": ablation_summary,
        "selection_decision": selection_decision,
    }
    return {
        "target_seed": target_seed,
        "tuning_seed": tuning_seed,
        "final_seed": final_seed,
        "run_config": run_config,
        "run_config_sha256": run_config_hash(run_config),
        "advance_to_multiseed": advance_variant,
        "threshold_by_variant_phase": threshold_by_variant_phase,
        "final_state_dicts": final_state_dicts,
        "pre_export_checks": pre_export_checks,
        "output_tables": output_tables,
        "ablation_summary": ablation_summary,
    }

In [6]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        if not np.isfinite(value):
            raise ValueError(f"Non-finite value cannot be serialized to JSON: {value}")
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return json_ready(value.item())
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {
            column: dtype_name(dtype) for column, dtype in table.dtypes.items()
        },
    }


def verify_published_run(published_run: dict[str, Any]) -> None:
    expected_seed = int(published_run["target_seed"])
    expected_manifest = published_run["manifest"]
    generation_root = Path(published_run["generation_root"])
    manifest_path = generation_root / "manifest.json"
    if not manifest_path.is_file():
        raise ValueError(f"Missing immutable manifest for seed {expected_seed}")
    actual_manifest = strict_json_loads(manifest_path.read_text(encoding="utf-8"))
    if actual_manifest != expected_manifest:
        raise ValueError(f"Immutable manifest mismatch for seed {expected_seed}")
    if actual_manifest.get("ablation_schema_version") != "dgd-ablation-v1":
        raise ValueError(f"Unexpected ablation schema for seed {expected_seed}")
    if actual_manifest.get("ablation_status") != "PASS":
        raise ValueError(f"Ablation status is not PASS for seed {expected_seed}")
    if int(actual_manifest.get("run_config", {}).get("seed", -1)) != expected_seed:
        raise ValueError(f"Manifest run_config.seed mismatch for seed {expected_seed}")
    if actual_manifest.get("run_config_sha256") != run_config_hash(
        actual_manifest["run_config"]
    ):
        raise ValueError(f"Run-config hash mismatch for seed {expected_seed}")
    immutable_manifest = resolve_inside(ARTIFACT_ROOT, actual_manifest["bundle_manifest"])
    if immutable_manifest != manifest_path.resolve():
        raise ValueError(f"Bundle-manifest path mismatch for seed {expected_seed}")
    checks = actual_manifest.get("checks")
    if not isinstance(checks, list) or not checks or not all(
        isinstance(row, dict) and row.get("status") == "PASS" for row in checks
    ):
        raise ValueError(f"Manifest contains a failed check for seed {expected_seed}")

    for name, artifact in actual_manifest.get("artifacts", {}).items():
        artifact_path = resolve_inside(ARTIFACT_ROOT, artifact["path"])
        artifact_path.resolve().relative_to(generation_root.resolve())
        if not artifact_path.is_file() or sha256_file(artifact_path) != artifact["sha256"]:
            raise ValueError(f"Artifact verification failed for seed {expected_seed}: {name}")
    expected_schema_names = set(published_run["output_table_names"])
    if set(actual_manifest.get("output_schemas", {})) != expected_schema_names:
        raise ValueError(f"Output-schema contract mismatch for seed {expected_seed}")
    expected_artifact_names = expected_schema_names | {
        "feature_contract",
        "final_model_checkpoints",
    }
    if set(actual_manifest.get("artifacts", {})) != expected_artifact_names:
        raise ValueError(f"Artifact contract mismatch for seed {expected_seed}")
    if actual_manifest.get("feature_contract_sha256") != actual_manifest["artifacts"][
        "feature_contract"
    ]["sha256"]:
        raise ValueError(f"Feature-contract hash mismatch for seed {expected_seed}")


def publish_seed_run(seed_run: dict[str, Any]) -> dict[str, Any]:
    target_seed = int(seed_run["target_seed"])
    bundle_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        + f"-s{target_seed}-"
        + uuid.uuid4().hex[:12]
    )
    staging_root = MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
    generation_root = MODEL_OUTPUT_ROOT / "generations" / bundle_id
    pointer_path = MODEL_OUTPUT_ROOT / "manifest.json"
    previous_pointer_text = (
        pointer_path.read_text(encoding="utf-8") if pointer_path.is_file() else None
    )
    generation_moved = False
    staging_root.mkdir(parents=True, exist_ok=False)
    output_tables = seed_run["output_tables"]
    output_artifacts: dict[str, Any] = {}

    try:
        for name, table in output_tables.items():
            staging_path = staging_root / f"{name}.csv"
            published_path = generation_root / f"{name}.csv"
            write_csv(staging_path, table)
            output_artifacts[name] = {
                "path": relative_output(published_path),
                "sha256": sha256_file(staging_path),
                "rows": len(table),
            }

        feature_contract_path = staging_root / "feature_contract.json"
        write_text(
            feature_contract_path,
            strict_json_dumps(json_ready(FEATURE_CONTRACT), indent=2, sort_keys=True)
            + "\n",
        )
        output_artifacts["feature_contract"] = {
            "path": relative_output(generation_root / "feature_contract.json"),
            "sha256": sha256_file(feature_contract_path),
        }

        checkpoint_path = staging_root / "final_models.pt"
        torch.save(
            {
                "schema_version": seed_run["run_config"]["schema_version"],
                "model_state_dicts": seed_run["final_state_dicts"],
                "shared_model_config": SHARED_MODEL_CONFIG,
                "feature_contract": FEATURE_CONTRACT,
                "threshold_by_variant_phase": {
                    f"{variant}|{phase}": threshold
                    for (variant, phase), threshold in seed_run[
                        "threshold_by_variant_phase"
                    ].items()
                },
                "advance_to_multiseed": seed_run["advance_to_multiseed"],
                "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            checkpoint_path,
        )
        output_artifacts["final_model_checkpoints"] = {
            "path": relative_output(generation_root / "final_models.pt"),
            "sha256": sha256_file(checkpoint_path),
        }

        manifest = {
            "ablation_schema_version": seed_run["run_config"]["schema_version"],
            "ablation_status": "PASS",
            "bundle_id": bundle_id,
            "bundle_manifest": relative_output(generation_root / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_protocol": {
                "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
                "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
                "pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            "run_config": seed_run["run_config"],
            "run_config_sha256": seed_run["run_config_sha256"],
            "feature_contract_sha256": sha256_file(feature_contract_path),
            "checks": seed_run["pre_export_checks"],
            "summary": {
                "variants": len(ABLATION_MATRIX),
                "advance_to_multiseed": seed_run["advance_to_multiseed"],
                "mean_validation_f1": float(
                    seed_run["ablation_summary"].loc[0, "mean_validation_f1"]
                ),
                "mean_validation_auc": float(
                    seed_run["ablation_summary"].loc[0, "mean_validation_auc"]
                ),
                "evaluation_prediction_rows": len(
                    output_tables["evaluation_predictions"]
                ),
            },
            "artifacts": output_artifacts,
            "output_schemas": {
                name: csv_schema(table) for name, table in output_tables.items()
            },
            "training_contract": {
                "protocol": "notebook 02 ml1m-coldstart-v1",
                "features": "same tuning-fit side-feature encoders across all variants",
                "thresholds": "selected on validation query rows only, per variant and phase",
                "selection": seed_run["run_config"]["selection_rule"],
            },
        }
        manifest = json_ready(manifest)
        manifest_text = strict_json_dumps(manifest, indent=2, sort_keys=True) + "\n"
        write_text(staging_root / "manifest.json", manifest_text)
        generation_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root.replace(generation_root)
        generation_moved = True

        published_run = {
            "target_seed": target_seed,
            "tuning_seed": int(seed_run["tuning_seed"]),
            "final_seed": int(seed_run["final_seed"]),
            "bundle_id": bundle_id,
            "advance_to_multiseed": seed_run["advance_to_multiseed"],
            "manifest": manifest,
            "manifest_path": str(generation_root / "manifest.json"),
            "generation_root": str(generation_root),
            "output_table_names": list(output_tables),
        }
        verify_published_run(published_run)
        write_text(pointer_path, manifest_text)
        if pointer_path.read_text(encoding="utf-8") != manifest_text:
            raise RuntimeError(f"Mutable pointer verification failed for seed {target_seed}")
        return published_run
    except Exception as error:
        if staging_root.exists():
            shutil.rmtree(staging_root, ignore_errors=True)
        if generation_moved:
            rollback_errors: list[str] = []
            try:
                if previous_pointer_text is None:
                    pointer_path.unlink(missing_ok=True)
                else:
                    write_text(pointer_path, previous_pointer_text)
            except Exception as rollback_error:
                rollback_errors.append(f"pointer restore: {rollback_error}")
            try:
                if generation_root.exists():
                    shutil.rmtree(generation_root)
            except Exception as rollback_error:
                rollback_errors.append(f"generation removal: {rollback_error}")
            if rollback_errors:
                raise RuntimeError(
                    f"Failed to roll back ablation generation {bundle_id}: "
                    + " | ".join(rollback_errors)
                ) from error
        raise

In [7]:
PUBLISHED_RUNS: list[dict[str, Any]] = []
RUN_REGISTRY_ROWS: list[dict[str, Any]] = []
RUN_FAILURE: Exception | None = None

for run_order, target_seed in enumerate(TARGET_SEEDS, start=1):
    seed_run: dict[str, Any] | None = None
    try:
        seed_run = build_seed_run(target_seed)
        published_run = publish_seed_run(seed_run)
        PUBLISHED_RUNS.append(published_run)
        RUN_REGISTRY_ROWS.append(
            {
                "run_order": run_order,
                "target_seed": target_seed,
                "run_config_seed": int(published_run["manifest"]["run_config"]["seed"]),
                "tuning_seed": published_run["tuning_seed"],
                "final_seed": published_run["final_seed"],
                "bundle_id": published_run["bundle_id"],
                "advance_to_multiseed": published_run["advance_to_multiseed"],
                "status": "PASS",
                "manifest": published_run["manifest_path"],
            }
        )
    except Exception as error:
        RUN_FAILURE = error
        RUN_REGISTRY_ROWS.append(
            {
                "run_order": run_order,
                "target_seed": target_seed,
                "run_config_seed": target_seed,
                "tuning_seed": target_seed + 1_000,
                "final_seed": target_seed + 10_000,
                "bundle_id": "",
                "advance_to_multiseed": "",
                "status": "BLOCKED",
                "manifest": "",
            }
        )
        break
    finally:
        seed_run = None
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

RUN_REGISTRY = pd.DataFrame(
    RUN_REGISTRY_ROWS,
    columns=[
        "run_order",
        "target_seed",
        "run_config_seed",
        "tuning_seed",
        "final_seed",
        "bundle_id",
        "advance_to_multiseed",
        "status",
        "manifest",
    ],
)

ABLATION_EXPORT_CHECKS: list[dict[str, Any]] = []


def final_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    ABLATION_EXPORT_CHECKS.append(
        {
            "check": name,
            "status": "PASS" if condition else "FAIL",
            "observed": observed,
            "expected": expected,
        }
    )


verified_seeds: list[int] = []
verification_errors: list[str] = []
for published_run in PUBLISHED_RUNS:
    try:
        verify_published_run(published_run)
        verified_seeds.append(int(published_run["target_seed"]))
    except Exception as error:
        verification_errors.append(
            f"seed {published_run['target_seed']}: {type(error).__name__}: {error}"
        )

successful_seeds = [
    int(row["target_seed"])
    for row in RUN_REGISTRY_ROWS
    if row["status"] == "PASS"
]
bundle_ids = [str(run["bundle_id"]) for run in PUBLISHED_RUNS]
manifest_seeds = [
    int(run["manifest"]["run_config"]["seed"]) for run in PUBLISHED_RUNS
]
selected_variants = [
    str(run["manifest"].get("summary", {}).get("advance_to_multiseed", ""))
    for run in PUBLISHED_RUNS
]
pointer_path = MODEL_OUTPUT_ROOT / "manifest.json"
pointer_matches_last_success = bool(PUBLISHED_RUNS) and pointer_path.is_file() and (
    pointer_path.read_bytes()
    == Path(PUBLISHED_RUNS[-1]["manifest_path"]).read_bytes()
)

final_check(
    "exact target seeds exported",
    successful_seeds == TARGET_SEEDS,
    successful_seeds,
    TARGET_SEEDS,
)
final_check(
    "one immutable generation per target seed",
    len(bundle_ids) == len(TARGET_SEEDS) and len(bundle_ids) == len(set(bundle_ids)),
    bundle_ids,
    f"{len(TARGET_SEEDS)} unique bundle ids",
)
final_check(
    "manifest seeds match target seeds",
    manifest_seeds == TARGET_SEEDS,
    manifest_seeds,
    TARGET_SEEDS,
)
final_check(
    "per-seed validation selections recorded",
    len(selected_variants) == len(TARGET_SEEDS) and all(selected_variants),
    dict(zip(manifest_seeds, selected_variants)),
    "one nonempty validation-selected variant per target seed; agreement not required",
)
final_check(
    "all manifests and artifact hashes verify",
    verified_seeds == TARGET_SEEDS and not verification_errors,
    {"verified_seeds": verified_seeds, "errors": verification_errors},
    {"verified_seeds": TARGET_SEEDS, "errors": []},
)
final_check(
    "mutable pointer matches last successful seed",
    pointer_matches_last_success,
    pointer_matches_last_success,
    True,
)
final_check(
    "no seed run failed",
    RUN_FAILURE is None,
    None if RUN_FAILURE is None else f"{type(RUN_FAILURE).__name__}: {RUN_FAILURE}",
    None,
)

ABLATION_PASS = all(row["status"] == "PASS" for row in ABLATION_EXPORT_CHECKS)
ABLATION_MANIFEST = PUBLISHED_RUNS[-1]["manifest"] if PUBLISHED_RUNS else None
ABLATION_POINTER = pointer_path

display(RUN_REGISTRY)
display(pd.DataFrame(ABLATION_EXPORT_CHECKS))
display(
    Markdown(
        "### Notebook 06 DGD ablation study: "
        + ("READY" if ABLATION_PASS else "BLOCKED")
    )
)

if not ABLATION_PASS:
    raise RuntimeError(
        "Ablation multi-seed run is BLOCKED; inspect RUN_REGISTRY and "
        "ABLATION_EXPORT_CHECKS"
    ) from RUN_FAILURE

display(
    Markdown(
        "**Next notebook:** notebook 07 should consume all per-seed bundles and WARN, "
        "rather than relabel or reselect, if validation-selected ablation variants differ."
    )
)

,run_order,target_seed,run_config_seed,tuning_seed,final_seed,bundle_id,advance_to_multiseed,status,manifest
0,1,2025,2025,3025,12025,20260717T021403351014-s2025-cc8981404b56,full_dgd,PASS,/kaggle/working/artifacts/models/ml-1m/dgd-abl...
1,2,7788,7788,8788,17788,20260717T021946974431-s7788-001c9cfe9c5b,full_dgd,PASS,/kaggle/working/artifacts/models/ml-1m/dgd-abl...
2,3,9999,9999,10999,19999,20260717T022529670475-s9999-7c3011e765ac,full_dgd,PASS,/kaggle/working/artifacts/models/ml-1m/dgd-abl...
3,4,3407,3407,4407,13407,20260717T023114264309-s3407-6803fb024589,full_dgd,PASS,/kaggle/working/artifacts/models/ml-1m/dgd-abl...
4,5,4517,4517,5517,14517,20260717T023658001768-s4517-5cf69cee9acf,full_dgd,PASS,/kaggle/working/artifacts/models/ml-1m/dgd-abl...


,check,status,observed,expected
0,exact target seeds exported,PASS,"[2025, 7788, 9999, 3407, 4517]","[2025, 7788, 9999, 3407, 4517]"
1,one immutable generation per target seed,PASS,"[20260717T021403351014-s2025-cc8981404b56, 202...",5 unique bundle ids
2,manifest seeds match target seeds,PASS,"[2025, 7788, 9999, 3407, 4517]","[2025, 7788, 9999, 3407, 4517]"
3,per-seed validation selections recorded,PASS,"{2025: 'full_dgd', 7788: 'full_dgd', 9999: 'fu...",one nonempty validation-selected variant per t...
4,all manifests and artifact hashes verify,PASS,"{'verified_seeds': [2025, 7788, 9999, 3407, 45...","{'verified_seeds': [2025, 7788, 9999, 3407, 45..."
5,mutable pointer matches last successful seed,PASS,True,True
6,no seed run failed,PASS,None,None


### Notebook 06 DGD ablation study: READY

**Next notebook:** notebook 07 should consume all per-seed bundles and WARN, rather than relabel or reselect, if validation-selected ablation variants differ.